orchrun 是 PyTorch 1.9.0+ 引入的分布式训练启动工具，取代了旧的 torch.distributed.launch。它提供了更简洁的命令行界面和更强大的功能，用于启动分布式训练任务。



| 特性         | 说明                            |
| ---------- | ----------------------------- |
| **弹性训练**   | 支持动态扩缩容节点                     |
| **容错机制**   | 自动重启失败的 worker                |
| **简化配置**   | 无需手动指定 `--nproc_per_node` 等参数 |
| **环境变量传递** | 自动处理 RANK、WORLD\_SIZE 等       |


## 单机多卡训练（最常用）

In [ ]:
# 使用 4 张 GPU 训练
torchrun --nproc_per_node=4 train.py

# 完整参数示例
torchrun \
    --nnodes=1 \                    # 节点数
    --nproc_per_node=4 \            # 每个节点的进程数（GPU数）
    --master_addr="localhost" \     # 主节点地址
    --master_port=29500 \           # 通信端口
    train.py \
    --batch_size 32 \
    --epochs 100

## 多级多卡训练

In [ ]:
# 在 Node 0 (主节点，IP: 192.168.1.10) 上运行：
torchrun \
    --nnodes=2 \
    --nproc_per_node=8 \
    --node_rank=0 \
    --master_addr="192.168.1.10" \
    --master_port=29500 \
    train.py

# 在 Node 1 上运行：
torchrun \
    --nnodes=2 \
    --nproc_per_node=8 \
    --node_rank=1 \
    --master_addr="192.168.1.10" \  # 主节点地址
    --master_port=29500 \
    train.py

## 弹性训练

In [ ]:
# 允许 1-4 个节点动态加入/离开
torchrun \
    --nnodes=1:4 \          # 最小1个节点，最大4个节点
    --nproc_per_node=8 \
    --max_restarts=3 \      # 最大重启次数
    --rdzv_id=job123 \      # 作业唯一标识
    --rdzv_backend=c10d \   # 协调后端
    --rdzv_endpoint="192.168.1.10:29500" \
    train.py

### 关键环境变量

torchrun 会自动设置以下环境变量，你可以在代码中使用：

In [ ]:
import os

os.environ["RANK"]          # 全局进程编号 (0 ~ world_size-1)
os.environ["LOCAL_RANK"]    # 本地 GPU 编号 (0 ~ nproc_per_node-1)
os.environ["WORLD_SIZE"]    # 总进程数
os.environ["MASTER_ADDR"]   # 主节点地址
os.environ["MASTER_PORT"]   # 通信端口